In [7]:
import pandas as pd
import numpy as np

df_real = pd.read_csv("water_leak_detection_1000_rows.csv", parse_dates=["Timestamp"])
df_gen  = pd.read_csv("water_leak_generated_5000_rows.csv",      parse_dates=["Timestamp"])

print("Real data shape:", df_real.shape)
print("Generated data shape:", df_gen.shape)
print("\nReal class dist:\n",  df_real[["Leak Status","Burst Status"]].value_counts())
print("\nGen class dist:\n",   df_gen[["Leak Status","Burst Status"]].value_counts())


Real data shape: (1000, 7)
Generated data shape: (5000, 7)

Real class dist:
 Leak Status  Burst Status
0            0               971
1            0                19
0            1                10
Name: count, dtype: int64

Gen class dist:
 Leak Status  Burst Status
0            0               3845
1            0                755
0            1                400
Name: count, dtype: int64


In [8]:
# Real data ends at: df_real["Timestamp"].max()
# We continue the generated timeline from that point + 5 min

real_end    = df_real["Timestamp"].max()
gen_start   = df_gen["Timestamp"].min()
time_offset = (real_end - gen_start) + pd.Timedelta(minutes=5)

df_gen["Timestamp"] = df_gen["Timestamp"] + time_offset

print("Real data ends  :", real_end)
print("Gen data starts :", df_gen["Timestamp"].min())
print("Gen data ends   :", df_gen["Timestamp"].max())


Real data ends  : 2024-01-04 11:15:00
Gen data starts : 2024-01-04 11:20:00
Gen data ends   : 2024-01-21 19:55:00


In [9]:
overlap = pd.merge(
    df_real[["Timestamp"]],
    df_gen[["Timestamp"]],
    on="Timestamp"
)
print("Overlapping timestamps:", len(overlap))   # must be → 0


Overlapping timestamps: 0


In [10]:
df_real["source"] = "real"
df_gen["source"]  = "generated"

df_merged = pd.concat([df_real, df_gen], ignore_index=True)

print("Merged shape:", df_merged.shape)
print("\nFinal class distribution:")
normal = ((df_merged["Leak Status"]==0) & (df_merged["Burst Status"]==0)).sum()
leak   = df_merged["Leak Status"].sum()
burst  = df_merged["Burst Status"].sum()
total  = len(df_merged)
print(f"  Normal : {normal:,}  ({normal/total*100:.1f}%)")
print(f"  Leak   : {leak:,}    ({leak/total*100:.1f}%)")
print(f"  Burst  : {burst:,}   ({burst/total*100:.1f}%)")


Merged shape: (6000, 8)

Final class distribution:
  Normal : 4,816  (80.3%)
  Leak   : 774    (12.9%)
  Burst  : 410   (6.8%)


In [11]:
df_shuffled = df_merged.sample(frac=1, random_state=42).reset_index(drop=True)

# Save full merged (with source column, useful for debugging)
df_shuffled.to_csv("water_leak_merged_6000.csv", index=False)

# Save model-ready version (drop source column for training)
df_model = df_shuffled.drop(columns=["source"])
df_model.to_csv("water_leak_train_ready.csv", index=False)

print("Saved: water_leak_merged_6000.csv  (with source tag)")
print("Saved: water_leak_train_ready.csv  (model-ready, no source tag)")


Saved: water_leak_merged_6000.csv  (with source tag)
Saved: water_leak_train_ready.csv  (model-ready, no source tag)
